# 01 - Setup (Task 1), baseline generation (Task 2), SFT (Task 3)

Accelerator: **GPU T4 x2**, Internet: **on**. Run all, then **Save Version**. ~2h.

In [ ]:
REPO_URL = "https://github.com/IsrarAliKhan/small-model-alignment-starter.git"  # edit if your fork lives elsewhere
%cd /kaggle/working
!git clone -q {REPO_URL} || git -C small-model-alignment-starter pull -q
%cd /kaggle/working/small-model-alignment-starter
!git log --oneline -1

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("CUDA devices:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Public adapter tests + stricter local tests (CPU-only, offline).

In [ ]:
!PYTHONPATH=. python -m pytest -q tests

## Task 2 - baseline generation on the fixed AlpacaEval subset (300 prompts, seed 42)

Greedy decoding, repetition_penalty 1.1, max_new_tokens 512 (same settings for every model). Sanity check on 3 prompts first.

In [ ]:
!python scripts/generate.py --model Qwen/Qwen2.5-0.5B --eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/sanity/base_alpaca.json --generator qwen2.5-0.5b-base --limit 3
import json; [print(r['instruction'][:100], '\n->', r['output'][:400], '\n') for r in json.load(open('outputs/sanity/base_alpaca.json'))]

In [ ]:
!python scripts/generate.py --model Qwen/Qwen2.5-0.5B --eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/generations/base_alpaca.json --generator qwen2.5-0.5b-base
!python scripts/generate.py --model Qwen/Qwen2.5-0.5B --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --output outputs/generations/base_hh.json --generator qwen2.5-0.5b-base

## Task 3 - SFT on 30k UltraChat single-turn examples (1k validation)

In [ ]:
!python scripts/prepare_data.py --task sft

In [ ]:
!python scripts/train_sft.py --output_dir outputs/sft

In [ ]:
!python scripts/plot_metrics.py --runs outputs/sft --keys loss --output outputs/plots/sft_loss.png
from IPython.display import Image; Image('outputs/plots/sft_loss.png')

In [ ]:
# Small zip of everything except checkpoints (metrics, configs, plots, generations).
!cd outputs && zip -qr ../sft_run_outputs.zip . -x "*/best/*" "*/final/*" "*/adapter/*" "*/trainer/*" && ls -la ../sft_run_outputs.zip
print("Now click 'Save Version' (Save & Run All or Quick Save) so outputs persist and can be attached later.")